# Boucle neuromodulée COMPLÈTE

On branche toute la boucle : **réservoir neuromodulé → predictive coding → tuyaux → inhibition latérale → décision**.

```
[ Chiffre ] → Physarum Neuromodulé ◄──┐
                   │                    │ Surprise S (module η et N_iter)
                   ▼                    │
             [ Signature z ]            │
                   │                    │
                   ▼                    │
        [ Predictive Coding ε = z - ẑ ]─┘
                   │
                   ▼
   [ Tuyaux + Inhibition Latérale ] → Décision
```

## 0. Imports

In [1]:
# Boucle neuromodulée COMPLÈTE — branchement intégral
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from recherche_agi import (load_mnist, train_readout, HybridBlobPredictive,
    NeuromodulatedReservoir, SynapticReservoir, lateral_inhibition,
    metabolic_n_iter, surprise_eta)

## 1. Données : MNIST

In [2]:
train_set, test_set = load_mnist()
print("Train :", len(train_set), "| Test :", len(test_set))

Train : 60000 | Test : 10000


## 2. Construction du prédicteur (prototypes par classe)

Pour calculer la **surprise** S = ‖z − ẑ‖, le réservoir neuromodulé a besoin d'un **prédicteur** ẑ. On construit des prototypes (moyenne des signatures par classe) comme prédiction de référence.

In [3]:
def extract(reservoir, dataset, n):
    X, y = [], []
    cnt = [0]*10
    for i in range(len(dataset)):
        l = int(dataset[i][1])
        if cnt[l] >= n//10: continue
        X.append(reservoir.signature(dataset[i][0].squeeze().numpy()))
        y.append(l); cnt[l] += 1
        if sum(cnt) >= n: break
    return np.array(X), np.array(y)

# Signatures de base pour les prototypes
base = SynapticReservoir(alpha=5.0, n_iter=10, n_zones=64, downscale=8, eta=0.1, gamma=0.1, beta=5.0)
Xtr0, ytr0 = extract(base, train_set, 300)
protos = {d: Xtr0[ytr0 == d].mean(axis=0) for d in range(10)}

def predictor(z):
    """Prédit la signature prototype la plus proche (pour calculer S)."""
    return min(protos.values(), key=lambda p: np.linalg.norm(p - z))

print("Prédicteur (prototypes par classe) prêt")

Prédicteur (prototypes par classe) prêt


## 3. Réservoir neuromodulé (plasticité + relaxation par surprise)

In [4]:
# Réservoir dont la dynamique est modulée par la surprise S
neu = NeuromodulatedReservoir(axes=('top_down','left_right'), n_zones=32, downscale=8,
                              predictor=predictor, n_max=20)

# Vérifier que la surprise module bien la dynamique
img = test_set[3][0].squeeze().numpy()   # un '3'
z = neu.signature(img)
print(f"Signature neuromodulée : {z.shape} | surprise S={neu.last_surprise:.3f}")
print(f"→ η = {surprise_eta(neu.last_surprise):.3f}, N_iter = {metabolic_n_iter(neu.last_surprise)}")

Signature neuromodulée : (64,) | surprise S=1.261
→ η = 0.731, N_iter = 50


## 4. Couche lue entraînée sur les signatures neuromodulées

In [5]:
Xtr, ytr = extract(neu, train_set, 200)
print(f"Signatures neuromodulées : {Xtr.shape}")
readout = train_readout(Xtr, ytr, n_classes=10, epochs=60)
with torch.no_grad():
    acc_ro = (readout(torch.tensor(Xtr, dtype=torch.float32)).argmax(1) == torch.tensor(ytr)).float().mean().item()
print(f"Acc couche lue (train) : {acc_ro:.3f}")

Signatures neuromodulées : (200, 64)


Acc couche lue (train) : 0.425


## 5. Predictive Coding + Blob (tuyaux) branchés sur le réservoir

In [6]:
hybrid = HybridBlobPredictive(readout, novelty_threshold=0.5, reservoir=neu)
cnt = [0]*10
for i in range(len(train_set)):
    l = int(train_set[i][1])
    if cnt[l] >= 3: continue
    hybrid.observe(train_set[i][0].squeeze().numpy(), label=l)
    cnt[l] += 1
    if sum(cnt) >= 30: break
print(f"Tuyaux créés (blob) : {len(hybrid.tubes)}")
print(f"Étiquettes des tuyaux : {[t.label for t in hybrid.tubes]}")

Tuyaux créés (blob) : 3
Étiquettes des tuyaux : [5, 4, 9]


## 6. Décision avec inhibition latérale (compétition corticale)

In [7]:
# Classification standard vs inhibition latérale
n_test = 100
correct_std = correct_lat = 0
exemple_activations = None
for i in range(n_test):
    img, label = test_set[i]
    img_np = img.squeeze().numpy()
    idx_std, _ = hybrid.classify(img_np)
    idx_lat, A = hybrid.classify_lateral(img_np, tau=0.5, method='softmax')
    if idx_std is not None and hybrid.tubes[idx_std].label == int(label): correct_std += 1
    if idx_lat is not None and hybrid.tubes[idx_lat].label == int(label): correct_lat += 1
    if i == 0: exemple_activations = A
print(f"Classification standard       : {correct_std}/{n_test} = {correct_std/n_test:.3f}")
print(f"Classification inhibition lat : {correct_lat}/{n_test} = {correct_lat/n_test:.3f}")
print(f"\nActivations compétitives (1er exemple) : {np.round(exemple_activations, 3)}")

Classification standard       : 19/100 = 0.190
Classification inhibition lat : 19/100 = 0.190

Activations compétitives (1er exemple) : [0.559 0.311 0.129]


## 7. Synthèse de la boucle complète

In [8]:
print("=== BOUCLE NEUROMODULÉE COMPLÈTE — SYNTHÈSE ===")
print(f"  Réservoir neuromodulé : surprise S module η et N_iter")
print(f"  Couche lue sur z : {acc_ro:.3f}")
print(f"  Tuyaux (blob) : {len(hybrid.tubes)}")
print(f"  Classification standard : {correct_std/n_test:.3f}")
print(f"  Classification inhibition latérale : {correct_lat/n_test:.3f}")
print()
print("Toute la boucle est branchée : Physarum neuromodulé → PC → tuyaux →")
print("inhibition latérale → décision. L'inhibition latérale rend la décision")
print("nette (WTA doux) ; son effet est visible quand plusieurs tuyaux hésitent.")

=== BOUCLE NEUROMODULÉE COMPLÈTE — SYNTHÈSE ===
  Réservoir neuromodulé : surprise S module η et N_iter
  Couche lue sur z : 0.425
  Tuyaux (blob) : 3
  Classification standard : 0.190
  Classification inhibition latérale : 0.190

Toute la boucle est branchée : Physarum neuromodulé → PC → tuyaux →
inhibition latérale → décision. L'inhibition latérale rend la décision
nette (WTA doux) ; son effet est visible quand plusieurs tuyaux hésitent.
